- ## Importing required libraries

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run "/Workspace/Users/abdulm63633@gmail.com/Ecommerce Lakehouse Project/01_setup_file/setup_utils"

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog", "ecommerce_lakehouse_project", "Catalog")
dbutils.widgets.text("data_source", "shipments", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")



In [0]:
df_silver = spark.sql(f"select * from {catalog}.{silver_schema}.{data_source}")
display(df_silver.limit(10))

In [0]:
df_gold = df_silver.select("shipment_id","delivery_date","shipment_status","warehouse")
df_gold.show()

In [0]:


if not (spark.catalog.tableExists(f"{catalog}.{gold_schema}.fact_{data_source}")):
    df_gold.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(
        f"{catalog}.{gold_schema}.fact_{data_source}"
    )
    print("sucessfully writed  data to delta location")
else:
    print("Doing upsert operation")
    delta_table = DeltaTable.forName(
        spark,
        f"{catalog}.{gold_schema}.fact_{data_source}"
    )

    delta_table.alias("target").merge(
        source=df_gold.alias("source"),
        condition="""
            target.shipment_id = source.shipment_id
        """
    ).whenMatchedUpdate(
        condition="""
        NOT (target.delivery_date <=> source.delivery_date)
        OR NOT (target.shipment_status <=> source.shipment_status)
        OR NOT (target.warehouse <=> source.warehouse)
    """,
        set={
             "delivery_date": "coalesce(source.delivery_date, target.delivery_date)",
            "shipment_status": "coalesce(source.shipment_status, target.shipment_status)",
            "warehouse": "coalesce(source.warehouse, target.warehouse)"
            
        }
    ).whenNotMatchedInsert(
        values={
            "shipment_id": "source.shipment_id",
            "delivery_date": "source.delivery_date",
            "shipment_status": "source.shipment_status",
            "warehouse": "source.warehouse"
        }
    ).execute()

